# Lab 3: Architectures in Practice

A convolution and a self-attention layer make different bets about what matters
in an image. You will write the three pieces those bets rest on: image
patchification, scaled dot-product attention, and the convolution output-size
rule. Then you put them to work in a controlled comparison of a Vision
Transformer against a small CNN, design a ViT to a budget, and change one part
of it.

**What you will do**

1. Turn an image into a sequence of patch tokens and count them.
2. Compute scaled dot-product attention and confirm it on a fixed probe.
3. Use the convolution output-size rule to size a CNN's classifier.
4. Design a ViT to a parameter budget, then change one architectural factor.

The red **Your Task** banners mark the parts you complete. Everything else is
provided and ready to run.


## At a glance

| | |
|---|---|
| Stack | PyTorch and torchvision; a GPU runtime is recommended |
| You write | `patchify`, `scaled_dot_product_attention`, `conv_output_size` |
| You submit | Five values, R1 through R5, from the final cell |
| GPU runtime | A few minutes of training in quick mode; longer on the full split |


<div style="border-left:6px solid #A31F34;background:#fff5f6;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#A31F34;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Big picture</div>
<p style="margin:6px 0 0;">The same picture can be read as a grid of local patches or a set of tokens that attend to each other. Build both readouts yourself, then let a fair comparison, not folklore, tell you where each one helps.</p>
</div>


## What is graded

R1 through R3 check your three functions on fixed inputs. R4 accepts your ViT
design as long as its parameter count lands in the stated range. R5 is a
pass-or-fail flag on your experimental setup. Measured CIFAR-10 accuracy is never
submitted; you read it to interpret your runs, and the graded numbers are the
same in quick and full mode.


<div style="border-left:6px solid #B45309;background:#fffbeb;color:#111827;padding:14px 18px;border-radius:8px;margin:18px 0;">
<div style="color:#B45309;font-weight:700;text-transform:uppercase;letter-spacing:0.06em;font-size:0.78rem;">Watch out</div>
<p style="margin:6px 0 0;">A short training budget can make either architecture look better by luck. Read the curves as evidence, not a verdict, and change only one factor at a time so a difference is traceable to that factor.</p>
</div>


In [ ]:
from __future__ import annotations

import math
import random
from dataclasses import asdict, dataclass, replace

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

SEED = 7960
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def reset_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


plt.rcParams.update({
    "figure.figsize": (7, 4),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "axes.prop_cycle": plt.cycler(
        color=["#A31F34", "#1D4ED8", "#059669", "#B45309", "#7C3AED"]
    ),
})

reset_seed()
print(f"PyTorch {torch.__version__}; device = {DEVICE}")


In [ ]:
QUICK_MODE = True
DATA_ROOT = "./data"

TRAIN_PER_CLASS = 400
VALIDATION_PER_CLASS = 100 if QUICK_MODE else 400
DEFAULT_BATCH_SIZE = 128
EPOCHS = 3 if QUICK_MODE else 8
NUM_WORKERS = 0 if QUICK_MODE else 2

print(f"quick_mode={QUICK_MODE}, epochs={EPOCHS}")


In [ ]:
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose(
    [
        transforms.RandomCrop(32, padding=3, padding_mode="reflect"),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ]
)
eval_transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)]
)

split_source = datasets.CIFAR10(DATA_ROOT, train=True, download=True)


def stratified_split(targets, train_per_class, validation_per_class, seed):
    target_tensor = torch.as_tensor(targets)
    generator = torch.Generator().manual_seed(seed)
    train_indices, validation_indices = [], []
    for class_id in range(10):
        class_indices = torch.where(target_tensor == class_id)[0]
        order = torch.randperm(len(class_indices), generator=generator)
        class_indices = class_indices[order]
        needed = train_per_class + validation_per_class
        train_indices.extend(class_indices[:train_per_class].tolist())
        validation_indices.extend(
            class_indices[train_per_class:needed].tolist()
        )
    return sorted(train_indices), sorted(validation_indices)


train_indices, validation_indices = stratified_split(
    split_source.targets, TRAIN_PER_CLASS, VALIDATION_PER_CLASS, SEED
)
assert set(train_indices).isdisjoint(validation_indices)

train_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=train_transform, download=False
)
eval_source = datasets.CIFAR10(
    DATA_ROOT, train=True, transform=eval_transform, download=False
)
train_set = Subset(train_source, train_indices)
train_eval_set = Subset(eval_source, train_indices)
validation_set = Subset(eval_source, validation_indices)


def seeded_loader(dataset, *, batch_size, shuffle, seed=SEED):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False,
        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
        generator=generator,
    )


def trainable_parameter_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


print("training examples:", len(train_indices),
      "validation examples:", len(validation_indices))


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 1</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>patchify</code></div>
</div>


Complete `patchify` so a `(batch, channels, H, W)` image becomes a
`(batch, num_patches, channels * patch * patch)` tensor of non-overlapping
patches, read row by row. A ViT then prepends one class token, so the sequence
entering the first block has `num_patches + 1` entries.


In [ ]:
def patchify(images, patch_size):
    """Return (batch, num_patches, channels*patch*patch) non-overlapping patches."""
    # STUDENT TASK 1: reshape into a (B, C, nh, p, nw, p) view, move the patch
    # grid next to the batch, then flatten each patch to a single vector.
    raise NotImplementedError("Return the (batch, num_patches, patch_dim) tensor.")


In [ ]:
patch_probe = patchify(torch.zeros(1, 3, 32, 32), patch_size=4)
num_patches = patch_probe.shape[1]
patch_token_count = num_patches + 1
patch_probe_contract = (
    tuple(patch_probe.shape) == (1, 64, 48)
    and patch_token_count == 65
)
assert patch_probe_contract, "Expected 64 patches of dimension 48, plus one class token."
print("Task 1 checks passed. Tokens entering the first block:", patch_token_count)


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 2</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>scaled_dot_product_attention</code></div>
</div>


Complete `scaled_dot_product_attention`. For queries `Q`, keys `K`, values `V`
with feature dimension `d`, return the output `softmax(QK^T / sqrt(d)) V` and the
attention weights. Use `key.transpose(-2, -1)` and `dim=-1` so the same function
works for a single head and for batched multi-head tensors.


<details style="border:1px solid #e5e7eb;border-radius:8px;padding:10px 14px;background:#f9fafb;color:#111827;margin:14px 0;">
<summary style="cursor:pointer;font-weight:600;">Which dimension does softmax go over?</summary>
<div style="margin-top:8px;">Take softmax over the last dimension, so each query's weights across the keys sum to one. Dividing by <code>sqrt(d)</code> before the softmax keeps the scores from saturating as <code>d</code> grows.</div>
</details>


In [ ]:
def scaled_dot_product_attention(query, key, value):
    """Return (output, attention_weights) for scaled dot-product attention."""
    # STUDENT TASK 2:
    #   d = query.shape[-1]
    #   scores = query @ key.transpose(-2, -1) / sqrt(d)
    #   weights = softmax(scores, dim=-1); output = weights @ value
    raise NotImplementedError("Return (output, weights).")


In [ ]:
attn_q = torch.tensor([[2 * math.log(3), 0.0, 0.0, 0.0]])
attn_k = torch.tensor([[1.0, 0.0, 0.0, 0.0], [-1.0, 0.0, 0.0, 0.0]])
attn_v = torch.tensor([[5.0, 0.0, 0.0, 0.0], [-5.0, 0.0, 0.0, 0.0]])
attn_out, attn_w = scaled_dot_product_attention(attn_q, attn_k, attn_v)
attention_probe_value = round(attn_out[0, 0].item(), 4)
attention_probe_contract = (
    tuple(attn_w.shape) == (1, 2)
    and abs(attn_w.sum().item() - 1.0) < 1e-6
    and abs(attn_w[0, 0].item() - 0.9) < 1e-4
    and abs(attention_probe_value - 4.0) < 1e-4
)
assert attention_probe_contract, "Check the sqrt(d) scaling, softmax, and weighted sum."
print("Task 2 checks passed. Attention probe output:", attention_probe_value)


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 3</div>
<div style="font-weight:700;font-size:1.2rem;">Write <code>conv_output_size</code></div>
</div>


Complete `conv_output_size`, the standard rule
`floor((W - K + 2P) / S) + 1`. The CNN below uses it to compute the flattened
feature dimension that feeds its linear classifier.


In [ ]:
def conv_output_size(input_size, kernel_size, stride, padding):
    """Return the output spatial size of one conv/pool stage."""
    # STUDENT TASK 3: implement floor((input - kernel + 2*padding)/stride) + 1
    raise NotImplementedError("Return the integer output size.")


In [ ]:
conv_size = 32
for kernel, stride, padding in [(3, 1, 1), (2, 2, 0), (3, 1, 1), (2, 2, 0)]:
    conv_size = conv_output_size(conv_size, kernel, stride, padding)
conv_feature_dim = 16 * conv_size * conv_size
conv_probe_contract = (conv_size == 8 and conv_feature_dim == 1024)
assert conv_probe_contract, "Two conv+pool stages take 32x32 to 8x8; 16*8*8 = 1024."
print("Task 3 checks passed. CNN classifier input dimension:", conv_feature_dim)


## Provided: model rails

The ViT calls your `patchify` and `scaled_dot_product_attention`; the CNN calls
your `conv_output_size`. Every run uses the same split and seed and records a
complete history.


In [ ]:
class CompactViT(nn.Module):
    def __init__(self, embed_dim, depth, heads, patch_size=4,
                 image_size=32, in_channels=3, num_classes=10, mlp_ratio=4):
        super().__init__()
        self.patch_size, self.heads = patch_size, heads
        patch_dim = in_channels * patch_size * patch_size
        num_patches = (image_size // patch_size) ** 2
        self.embed = nn.Linear(patch_dim, embed_dim)
        self.cls = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.blocks = nn.ModuleList(
            nn.ModuleDict({
                "norm1": nn.LayerNorm(embed_dim),
                "qkv": nn.Linear(embed_dim, 3 * embed_dim),
                "proj": nn.Linear(embed_dim, embed_dim),
                "norm2": nn.LayerNorm(embed_dim),
                "mlp": nn.Sequential(
                    nn.Linear(embed_dim, mlp_ratio * embed_dim),
                    nn.GELU(),
                    nn.Linear(mlp_ratio * embed_dim, embed_dim),
                ),
            })
            for _ in range(depth)
        )
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, images):
        # Your patchify turns the image into tokens; prepend a class token and
        # add position embeddings, then run each Transformer block.
        tokens = self.embed(patchify(images, self.patch_size))
        cls = self.cls.expand(tokens.shape[0], -1, -1)
        tokens = torch.cat([cls, tokens], dim=1) + self.pos
        for block in self.blocks:
            x = block["norm1"](tokens)
            q, k, v = block["qkv"](x).chunk(3, dim=-1)
            # Split the embedding into heads, then call your attention per head.
            b, length, dim = q.shape
            head_dim = dim // self.heads
            shape = (b, length, self.heads, head_dim)
            q = q.reshape(shape).transpose(1, 2)
            k = k.reshape(shape).transpose(1, 2)
            v = v.reshape(shape).transpose(1, 2)
            attended, _ = scaled_dot_product_attention(q, k, v)
            attended = attended.transpose(1, 2).reshape(b, length, dim)
            tokens = tokens + block["proj"](attended)
            tokens = tokens + block["mlp"](block["norm2"](tokens))
        return self.head(self.norm(tokens)[:, 0])


class SmallCNN(nn.Module):
    def __init__(self, channels=16, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, channels, 3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(channels, channels, 3, stride=1, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        size = 32
        for kernel, stride, padding in [(3, 1, 1), (2, 2, 0), (3, 1, 1), (2, 2, 0)]:
            size = conv_output_size(size, kernel, stride, padding)
        self.classifier = nn.Linear(channels * size * size, num_classes)

    def forward(self, images):
        return self.classifier(self.features(images).flatten(1))


@dataclass(frozen=True)
class ArchitectureExperiment:
    architecture: str = "vit"
    embed_dim: int = 64
    depth: int = 2
    heads: int = 4
    patch_size: int = 4
    use_augmentation: bool = False
    learning_rate: float = 3e-4
    batch_size: int = DEFAULT_BATCH_SIZE
    epochs: int = EPOCHS
    seed: int = SEED


def changed_config_fields(reference, candidate):
    reference_fields = asdict(reference)
    candidate_fields = asdict(candidate)
    return {
        name for name in reference_fields
        if reference_fields[name] != candidate_fields[name]
    }


def history_is_complete(record):
    history = record["history"]
    epochs = record["config"]["epochs"]
    return all(
        len(history[metric]) == epochs
        for metric in (
            "train_loss", "train_accuracy",
            "validation_loss", "validation_accuracy",
        )
    )


def assemble_model(config):
    reset_seed(config.seed)
    if config.architecture == "cnn":
        return SmallCNN()
    return CompactViT(
        embed_dim=config.embed_dim, depth=config.depth,
        heads=config.heads, patch_size=config.patch_size,
    )


@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()
    loss_fn = nn.CrossEntropyLoss(reduction="sum")
    total_loss = total_correct = total_examples = 0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        total_loss += loss_fn(logits, labels).item()
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += labels.numel()
    return total_loss / total_examples, total_correct / total_examples


def run_experiment(label, config):
    # Train one architecture (a ViT built from your patchify and attention, or the
    # CNN sized by your conv_output_size) and return its config and full history.
    model = assemble_model(config).to(DEVICE)
    source = train_set if config.use_augmentation else train_eval_set
    fit_loader = seeded_loader(
        source, batch_size=config.batch_size, shuffle=True, seed=config.seed
    )
    train_eval_loader = seeded_loader(
        train_eval_set, batch_size=config.batch_size, shuffle=False,
        seed=config.seed,
    )
    validation_loader = seeded_loader(
        validation_set, batch_size=config.batch_size, shuffle=False,
        seed=config.seed,
    )
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)
    loss_fn = nn.CrossEntropyLoss()
    history = {
        "train_loss": [], "train_accuracy": [],
        "validation_loss": [], "validation_accuracy": [],
    }
    for epoch in range(1, config.epochs + 1):
        model.train()
        for images, labels in fit_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            loss = loss_fn(model(images), labels)
            loss.backward()
            optimizer.step()
        train_loss, train_accuracy = evaluate_model(model, train_eval_loader)
        validation_loss, validation_accuracy = evaluate_model(
            model, validation_loader
        )
        history["train_loss"].append(train_loss)
        history["train_accuracy"].append(train_accuracy)
        history["validation_loss"].append(validation_loss)
        history["validation_accuracy"].append(validation_accuracy)
        print(f"{label:>24} | epoch {epoch}/{config.epochs} | "
              f"val acc {validation_accuracy:.2%}")
    record = {"label": label, "config": asdict(config), "history": history}
    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return record


baseline_config = ArchitectureExperiment()
baseline_result = run_experiment("vit baseline", baseline_config)
cnn_config = replace(baseline_config, architecture="cnn")
cnn_result = run_experiment("cnn baseline", cnn_config)


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 4</div>
<div style="font-weight:700;font-size:1.2rem;">Design a ViT</div>
</div>


Choose `embed_dim` (a positive multiple of the head count, 4) so the ViT has
**200,000 to 1,500,000** trainable parameters. Larger widths add capacity but
also cost and overfitting risk.


In [ ]:
# STUDENT TASK 4: pick embed_dim inside the budget (a multiple of 4).
embed_dim = 0

if embed_dim <= 0 or embed_dim % 4 != 0:
    raise ValueError("Choose a positive embed_dim divisible by 4 (the head count).")
learner_config = replace(baseline_config, embed_dim=embed_dim)
learner_vit = assemble_model(learner_config)
learner_vit_parameter_count = trainable_parameter_count(learner_vit)
learner_parameter_count = learner_vit_parameter_count
if not (200_000 <= learner_parameter_count <= 1_500_000):
    raise ValueError("Choose embed_dim so the ViT has 200,000-1,500,000 parameters.")
print(f"learner ViT parameters: {learner_parameter_count:,}")


In [ ]:
learner_result = run_experiment("learner vit design", learner_config)
for record in (baseline_result, cnn_result, learner_result):
    epochs = np.arange(1, len(record["history"]["validation_accuracy"]) + 1)
    plt.plot(epochs, record["history"]["validation_accuracy"],
             marker="o", label=record["label"])
plt.xlabel("epoch"); plt.ylabel("val accuracy"); plt.ylim(0, 1)
plt.legend(); plt.tight_layout(); plt.show()


<div style="border-left:5px solid #A31F34;padding-left:12px;margin:26px 0 6px;">
<div style="color:#A31F34;font-weight:700;font-size:0.72rem;letter-spacing:0.11em;">YOUR TASK 5</div>
<div style="font-weight:700;font-size:1.2rem;">Choose one architectural variation</div>
</div>


Predict what will change, then choose exactly one single-factor change from the
ViT baseline: larger patches (`patch_8`), one more block (`deeper`), or fewer
attention heads (`fewer_heads`). The contract permits exactly one changed field.


In [ ]:
# STUDENT TASK 5: choose "patch_8", "deeper", or "fewer_heads".
VARIATION_CHOICE = ""

VARIATION_OVERRIDES = {
    "patch_8": {"patch_size": 8},
    "deeper": {"depth": 3},
    "fewer_heads": {"heads": 2},
}
VARIATION_FIELDS = {
    "patch_8": "patch_size",
    "deeper": "depth",
    "fewer_heads": "heads",
}
VARIATION_LABELS = {
    "patch_8": "chosen variation: patch 8",
    "deeper": "chosen variation: deeper",
    "fewer_heads": "chosen variation: fewer heads",
}
if VARIATION_CHOICE not in VARIATION_OVERRIDES:
    raise ValueError("Choose patch_8, deeper, or fewer_heads.")
variation_config = replace(
    baseline_config, **VARIATION_OVERRIDES[VARIATION_CHOICE]
)
assert changed_config_fields(baseline_config, variation_config) == {
    VARIATION_FIELDS[VARIATION_CHOICE]
}


In [ ]:
variation_result = run_experiment(
    VARIATION_LABELS[VARIATION_CHOICE], variation_config
)
for record in (baseline_result, variation_result):
    epochs = np.arange(1, len(record["history"]["validation_accuracy"]) + 1)
    plt.plot(epochs, record["history"]["validation_accuracy"],
             marker="o", label=record["label"])
plt.xlabel("epoch"); plt.ylabel("val accuracy"); plt.ylim(0, 1)
plt.legend(); plt.tight_layout(); plt.show()


## Pause and reflect (ungraded)

Which comparison most changed your prediction? Name one visible curve pattern,
one caveat that keeps the conclusion tentative under this short budget, and one
single-factor experiment you would run next. This is a place to pause, not a
response field.


## Report values

Run the cell below after completing all five tasks and all four runs. R1-R3
verify your three functions on fixed probes. R4 is **your** valid ViT design's
parameter count in the 200,000-1,500,000 range. R5 verifies the disjoint split,
single-factor comparisons, and complete histories. Enter R1-R5 in the first Lab
3 submission problem. No measured accuracy is submitted.


In [ ]:
run_config_pairs = (
    (baseline_result, baseline_config),
    (cnn_result, cnn_config),
    (learner_result, learner_config),
    (variation_result, variation_config),
)

workflow_contract = int(
    set(train_indices).isdisjoint(validation_indices)
    and len({record["label"] for record, _ in run_config_pairs}) == 4
    and all(
        record["config"] == asdict(config)
        for record, config in run_config_pairs
    )
    and all(history_is_complete(record) for record, _ in run_config_pairs)
    and changed_config_fields(baseline_config, cnn_config) == {"architecture"}
    and changed_config_fields(baseline_config, learner_config) == {"embed_dim"}
    and changed_config_fields(baseline_config, variation_config)
        == {VARIATION_FIELDS[VARIATION_CHOICE]}
    and learner_parameter_count == learner_vit_parameter_count
    and 200_000 <= learner_parameter_count <= 1_500_000
)

probe_token_count = patch_token_count if patch_probe_contract else -1
probe_attention_value = (
    round(attention_probe_value, 4) if attention_probe_contract else -1.0
)
probe_conv_dim = conv_feature_dim if conv_probe_contract else -1

report_values = {
    "R1: ViT patch-plus-class token count": probe_token_count,
    "R2: scaled dot-product attention probe output": probe_attention_value,
    "R3: CNN classifier input dimension": probe_conv_dim,
    "R4: learner-designed ViT parameter count": learner_parameter_count,
    "R5: experimental workflow contract": workflow_contract,
}
assert probe_token_count == 65
assert probe_attention_value == 4.0
assert probe_conv_dim == 1024
assert 200_000 <= learner_parameter_count <= 1_500_000
assert workflow_contract == 1

print("LAB 3 REPORT VALUES")
for label, value in report_values.items():
    print(f"{label}: {value}")
